# Clay v1.5 Inference and Similarity Search with LanceDB

This notebook demonstrates an end-to-end workflow for generating embeddings
from satellite imagery and using them for vector similarity search:

1. Fetch Sentinel-2 imagery over Lisbon, Portugal from the Element84 STAC catalog
2. Tile the imagery into a grid of 256x256 chips
3. Generate a CLS embedding for each chip using the Clay v1.5 encoder
4. Save embeddings to GeoParquet
5. Load into LanceDB for fast similarity search
6. Query and visualize similar chips

Lisbon provides excellent variety: urban fabric, the Tagus River, parks,
industrial zones, and agricultural land on the south bank.

**Requirements**: `pip install claymodel stackstac pystac-client geopandas matplotlib lancedb einops`

In [ ]:
%pip install -q lancedb

In [ ]:
import math
import os
import random

import geopandas as gpd
import lancedb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pystac_client
import stackstac
import torch
from rasterio.enums import Resampling
from shapely import Point
from torchvision.transforms import v2

from claymodel.api import _bundled_metadata_path, load_metadata
from claymodel.module import ClayMAEModule

## Configuration

We target Lisbon, Portugal for its diverse land cover: dense urban fabric,
the wide Tagus River estuary, parks, and agricultural fields on the margins.

In [ ]:
# Lisbon area — urban, water, vegetation mix
lat, lon = 38.72, -9.14
PLATFORM = "sentinel-2-l2a"
DATE = "2023-06-01"
DATE_RANGE = "2023-06-01/2023-06-30"
CHIP_SIZE = 256
GSD = 10

# 4x4 grid = 16 chips covering a 1024x1024 pixel area (10.24 km x 10.24 km)
GRID_N = 4

# Sentinel-2 bands matching metadata.yaml band_order
S2_BANDS = [
    "blue",
    "green",
    "red",
    "rededge1",
    "rededge2",
    "rededge3",
    "nir",
    "nir08",
    "swir16",
    "swir22",
]

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {device}")

## Fetch Sentinel-2 imagery from STAC

We search for a low-cloud scene and download all 10 bands at 10 m resolution
over a 1024x1024 pixel area (4x4 grid of 256-pixel chips).

In [ ]:
STAC_API = "https://earth-search.aws.element84.com/v1"
catalog = pystac_client.Client.open(STAC_API)

search = catalog.search(
    collections=[PLATFORM],
    datetime=DATE_RANGE,
    bbox=(lon - 0.1, lat - 0.1, lon + 0.1, lat + 0.1),
    max_items=10,
    query={"eo:cloud_cover": {"lt": 10}},
)

items = search.item_collection()
print(f"Found {len(items)} items with <10% cloud cover")

# Use the first item
item = items[0]
print(f"Using: {item.id} ({item.datetime.date()})")

In [ ]:
# Project point to UTM and create bounds for the full grid
epsg = int(item.properties["proj:code"].replace("EPSG:", ""))

poidf = gpd.GeoDataFrame(
    pd.DataFrame(), crs="EPSG:4326", geometry=[Point(lon, lat)]
).to_crs(epsg)
coords = poidf.iloc[0].geometry.coords[0]

total_pixels = CHIP_SIZE * GRID_N  # 1024
half_extent = (total_pixels * GSD) // 2

bounds = (
    coords[0] - half_extent,
    coords[1] - half_extent,
    coords[0] + half_extent,
    coords[1] + half_extent,
)

print(
    f"Fetching {total_pixels}x{total_pixels} pixels"
    f" = {GRID_N}x{GRID_N} chips of {CHIP_SIZE}x{CHIP_SIZE}"
)

In [ ]:
# Download all 10 Sentinel-2 bands
stack = stackstac.stack(
    [item],
    bounds=bounds,
    snap_bounds=False,
    epsg=epsg,
    resolution=GSD,
    dtype="float64",
    rescale=False,
    fill_value=0,
    assets=S2_BANDS,
    resampling=Resampling.nearest,
)
stack = stack.compute()
print(f"Stack shape: {stack.shape}  (time, band, y, x)")

In [ ]:
# Show the full area as RGB with chip grid overlay
rgb = stack.sel(band=["red", "green", "blue"]).values[0]

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(np.clip(rgb.transpose(1, 2, 0) / 3000, 0, 1))
ax.set_title(
    f"Lisbon area -- {GRID_N}x{GRID_N} chip grid ({total_pixels}x{total_pixels} px)"
)
ax.set_axis_off()

# Draw chip grid lines
for i in range(1, GRID_N):
    ax.axhline(y=i * CHIP_SIZE, color="white", linewidth=0.5, alpha=0.7)
    ax.axvline(x=i * CHIP_SIZE, color="white", linewidth=0.5, alpha=0.7)

plt.tight_layout()

## Tile into 256x256 chips

We slice the large image into non-overlapping chips. Each chip will
be independently embedded by the Clay encoder.

In [ ]:
# Extract the image as numpy: [band, H, W]
full_image = stack.values[0]  # first (only) timestep

chips = []
chip_positions = []

for row in range(GRID_N):
    for col in range(GRID_N):
        y0, x0 = row * CHIP_SIZE, col * CHIP_SIZE
        chip = full_image[:, y0 : y0 + CHIP_SIZE, x0 : x0 + CHIP_SIZE]
        chips.append(chip)
        chip_positions.append((row, col))

print(f"Created {len(chips)} chips of shape {chips[0].shape}  (band, H, W)")

In [ ]:
# Visualize all chips as RGB (bands: blue=0, green=1, red=2)
fig, axes = plt.subplots(GRID_N, GRID_N, figsize=(12, 12))

for idx, (chip, (r, c)) in enumerate(zip(chips, chip_positions)):
    rgb = chip[[2, 1, 0]].transpose(1, 2, 0)  # red, green, blue -> HWC
    axes[r, c].imshow(np.clip(rgb / 3000, 0, 1))
    axes[r, c].set_title(f"Chip {idx}", fontsize=8)
    axes[r, c].set_axis_off()

plt.suptitle("All chips (RGB)", fontsize=13)
plt.tight_layout()

## Load the Clay v1.5 model

We load with `mask_ratio=0.0` and `shuffle=False` so the encoder returns
all 1024 patch tokens in spatial order (no random masking).

In [ ]:
# Find or download checkpoint
ckpt = "clay-v1.5.ckpt"
if not os.path.exists(ckpt):
    ckpt = "../../clay-v1.5.ckpt"
if not os.path.exists(ckpt):
    print("Downloading checkpoint...")
    os.system(
        "wget -q https://huggingface.co/made-with-clay/Clay/resolve/main/"
        "v1.5/clay-v1.5.ckpt -O clay-v1.5.ckpt"
    )
    ckpt = "clay-v1.5.ckpt"

In [ ]:
torch.set_default_device(device)

metadata = load_metadata()

model = ClayMAEModule.load_from_checkpoint(
    ckpt,
    metadata_path=_bundled_metadata_path(),
    mask_ratio=0.0,
    shuffle=False,
)
model.eval()
model = model.to(device)
print(f"Model loaded on {device}")

## Generate embeddings for all chips

For each chip we build a datacube dict with normalized pixels, wavelengths,
GSD, and cyclic time/location encodings. The encoder returns
`[1, 1+1024, 1024]` -- we extract the CLS token at position 0 as the
single 1024-d embedding that summarizes the entire chip.

In [ ]:
def normalize_timestamp(date):
    """Encode date as cyclic (sin, cos) features for week and hour."""
    week = date.isocalendar().week * 2 * np.pi / 52
    hour = date.hour * 2 * np.pi / 24
    return (math.sin(week), math.cos(week)), (math.sin(hour), math.cos(hour))


def normalize_latlon(lat, lon):
    """Encode lat/lon as (sin, cos) features."""
    lat_r = lat * np.pi / 180
    lon_r = lon * np.pi / 180
    return (math.sin(lat_r), math.cos(lat_r)), (math.sin(lon_r), math.cos(lon_r))


def make_datacube(chip_pixels, bands, platform, metadata, lat, lon, date, device):
    """Build a datacube dict for a single chip.

    Parameters
    ----------
    chip_pixels : np.ndarray  [C, H, W]
    bands : list[str]  band names matching metadata.yaml
    platform : str
    metadata : dict
    lat, lon : float
    date : datetime
    device : torch.device
    """
    sensor = metadata[platform]

    mean = [sensor.bands.mean[b] for b in bands]
    std = [sensor.bands.std[b] for b in bands]
    waves = [sensor.bands.wavelength[b] for b in bands]

    transform = v2.Compose([v2.Normalize(mean=mean, std=std)])
    pixels = torch.from_numpy(chip_pixels.astype(np.float32)).unsqueeze(0)
    pixels = transform(pixels)

    time_enc = normalize_timestamp(date)
    latlon_enc = normalize_latlon(lat, lon)

    return {
        "pixels": pixels.to(device),
        "time": torch.tensor(
            [list(time_enc[0]) + list(time_enc[1])],
            dtype=torch.float32,
            device=device,
        ),
        "latlon": torch.tensor(
            [list(latlon_enc[0]) + list(latlon_enc[1])],
            dtype=torch.float32,
            device=device,
        ),
        "gsd": torch.tensor(sensor.gsd, device=device),
        "waves": torch.tensor(waves, device=device),
    }

In [ ]:
from datetime import datetime

date = datetime.strptime(DATE, "%Y-%m-%d")
bands = list(stack.band.values)

embeddings = []
chip_rgbs = []  # Store RGB arrays for visualization

for i, chip in enumerate(chips):
    datacube = make_datacube(chip, bands, PLATFORM, metadata, lat, lon, date, device)

    with torch.no_grad():
        unmsk_patch, *_ = model.model.encoder(datacube)
        cls_embedding = unmsk_patch[:, 0, :].cpu().numpy().squeeze()  # [1024]

    embeddings.append(cls_embedding)

    # Keep RGB thumbnail for later visualization
    rgb = chip[[2, 1, 0]].transpose(1, 2, 0)  # red, green, blue -> HWC
    chip_rgbs.append(np.clip(rgb / 3000, 0, 1))

    if (i + 1) % 4 == 0:
        print(f"  Embedded {i + 1}/{len(chips)} chips")

print(f"\nGenerated {len(embeddings)} embeddings of dimension {embeddings[0].shape[0]}")

## Save embeddings to GeoParquet

We store each chip's embedding alongside its grid position and a point
geometry. This GeoParquet file can be shared, loaded into QGIS, or
ingested into any vector database.

In [ ]:
outdir = "/tmp/clay-inference"
os.makedirs(outdir, exist_ok=True)

records = []
for i, emb in enumerate(embeddings):
    row, col = chip_positions[i]
    records.append(
        {
            "chip_id": i,
            "row": row,
            "col": col,
            "date": DATE,
            "embedding": emb.tolist(),
            "geometry": Point(lon, lat),  # same center for all chips in this demo
        }
    )

gdf = gpd.GeoDataFrame(records, geometry="geometry", crs="EPSG:4326")
parquet_path = os.path.join(outdir, "lisbon_embeddings.parquet")
gdf.to_parquet(path=parquet_path, compression="ZSTD")
print(f"Saved {len(gdf)} embeddings to {parquet_path}")

## Similarity search with LanceDB

LanceDB provides fast vector similarity search backed by the Lance columnar
format. We load our CLS embeddings into a table and query for the most
similar chips by L2 distance.

In [ ]:
db = lancedb.connect(os.path.join(outdir, "lancedb"))

# Build records for LanceDB (vector column must be named "vector")
lance_records = []
for i, emb in enumerate(embeddings):
    row, col = chip_positions[i]
    lance_records.append(
        {
            "vector": emb,
            "chip_id": i,
            "row": row,
            "col": col,
        }
    )

tbl = db.create_table("embeddings", data=lance_records, mode="overwrite")
n_vecs = len(lance_records)
dim = embeddings[0].shape[0]
print(f"Created LanceDB table with {n_vecs} vectors of dim {dim}")

## Query for similar chips

We pick a query chip and find the most similar chips in embedding space.
The first result is the query itself (distance = 0). Similar chips should
share land-cover characteristics (both water, both urban, etc.).

In [ ]:
# Pick a query chip (use seed for reproducibility)
random.seed(42)
query_idx = random.randint(0, len(embeddings) - 1)
query_vec = embeddings[query_idx]

# Search for top-6 most similar
n_results = min(6, len(embeddings))
results = tbl.search(query=query_vec).limit(n_results).to_pandas()

qr, qc = chip_positions[query_idx]
print(f"Query: chip {query_idx}  (row={qr}, col={qc})")
print(f"\nTop {n_results} most similar chips:")
print(results[["chip_id", "row", "col", "_distance"]].to_string(index=False))

### Visualize search results

The query chip on the left, followed by closest matches sorted by distance.
Visually similar land cover types should cluster together in embedding space.

In [ ]:
fig, axes = plt.subplots(1, n_results, figsize=(3 * n_results, 3))

for i, (ax, (_, row)) in enumerate(zip(axes, results.iterrows())):
    cid = int(row["chip_id"])
    ax.imshow(chip_rgbs[cid])

    if i == 0:
        ax.set_title(f"Query (chip {cid})", fontweight="bold", fontsize=9)
    else:
        ax.set_title(f"#{i}: chip {cid}\nd={row['_distance']:.1f}", fontsize=9)
    ax.set_axis_off()

plt.suptitle("Similarity search results (nearest embeddings)", fontsize=13)
plt.tight_layout()

## Pairwise cosine similarity heatmap

Computing similarity between all pairs of chip embeddings reveals the
structure of the embedding space. Chips with similar land cover should
have high cosine similarity (near 1.0).

In [ ]:
from torch.nn.functional import cosine_similarity

emb_tensor = torch.tensor(np.stack(embeddings))  # [N, D]
n = emb_tensor.shape[0]

# Pairwise cosine similarity
sim_matrix = np.zeros((n, n))
for i in range(n):
    sim_matrix[i] = cosine_similarity(emb_tensor[i : i + 1], emb_tensor, dim=1).numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap
im = ax1.imshow(sim_matrix, cmap="RdYlBu_r", vmin=0.5, vmax=1.0)
ax1.set_xlabel("Chip ID")
ax1.set_ylabel("Chip ID")
ax1.set_title("Pairwise cosine similarity")
plt.colorbar(im, ax=ax1, label="Cosine similarity")

# Thumbnail strip for reference
for idx in range(n):
    inset = ax2.inset_axes([idx / n, 0, 1 / n, 1])
    inset.imshow(chip_rgbs[idx])
    inset.set_title(str(idx), fontsize=6)
    inset.set_axis_off()
ax2.set_axis_off()
ax2.set_title("Chip thumbnails (by ID)")

plt.tight_layout()

## Summary

This notebook demonstrated the full inference-to-search pipeline:

1. **Fetched** 10-band Sentinel-2 imagery over Lisbon from a public STAC catalog
2. **Tiled** a 1024x1024 pixel area into 16 non-overlapping 256x256 chips
3. **Generated** a 1024-d CLS embedding per chip using the Clay v1.5 encoder
4. **Saved** embeddings to GeoParquet for portable storage
5. **Loaded** embeddings into LanceDB for vector similarity search
6. **Queried** for similar chips and verified that land-cover similarity
   correlates with embedding distance

The same workflow scales to thousands of chips across large regions. For
production use, consider:
- Batching chips for GPU-efficient inference
- Adding spatial metadata (per-chip bounding box) to the parquet/LanceDB records
- Using pre-computed embeddings from [source.coop](https://source.coop/clay/)
  instead of running inference locally

See `embeddings.ipynb` for a deeper look at patch-level spatial embeddings
and `wall-to-wall.ipynb` for PCA analysis over large areas.